# Data Analytics — Conversion Rate EDA

---

## Objective

This analysis examines a multi-channel marketing campaign dataset to understand what drives conversions and where campaign spend is most effective. The dataset captures performance across a range of campaign types, target audiences, and distribution channels, making it possible to compare conversion rates, return on investment, and acquisition costs in a structured way. The objective is to identify which combinations of campaign type, channel, and customer segment produce the strongest results and to surface any meaningful patterns between engagement volume and actual conversion performance.

## Data

The dataset contains campaign-level records with both configuration and outcome fields. On the configuration side, each record captures the campaign's type, target audience, channel used, customer segment, duration, and language. On the outcome side, it includes clicks, impressions, engagement score, conversion rate, acquisition cost, ROI, and the campaign date. Together these fields support both a top-level view of campaign effectiveness and a more granular breakdown by channel and segment. The dataset contains no missing values and required only deduplication before analysis.

In [ ]:
#Install needed libraries
!pip install prophet
!pip install chardet

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from prophet import Prophet

In [ ]:
#Load the data
campaign_df = pd.read_csv('marketing_campaign_dataset.csv')
campaign_df.head()

#Check for missing data

In [ ]:
campaign_df.isnull().sum()

#Describe the data

In [ ]:
# Display the basic statistics of numerical columns.
campaign_df.describe()

In [ ]:
#Explore the structure of the data
campaign_df.info()

There is no null cells present.

In [ ]:
campaign_df.dtypes

In [ ]:
campaign_df.shape

#Data cleaning

In [ ]:
#Deleting unnecessary rows to reduce the size of the dataset.
campaign_df.duplicated().sum()

In [ ]:
campaign_df.drop_duplicates(keep = 'first', ignore_index = True, inplace=True)

for col in campaign_df.columns:
    unique_values = campaign_df[campaign_df.duplicated(keep=False)][col].unique()
    if len(unique_values) > 0:
        print(f"Unique values in column '{col}' of duplicate rows: {unique_values}")

num_duplicates = campaign_df.duplicated().sum()
print(f"Number of duplicate rows: {num_duplicates}")

#Check for Outliers

In [ ]:
list1 = ['Clicks', 'Engagement_Score', 'Impressions']
for i in list1:
    print(str(i)+': ')
    ax = sns.boxplot(x=campaign_df[str(i)])
    plt.show()

#Exploratory Data Analysis

#Exploratory Data Analysis

##Conversion Rate by Campaign Type

In [ ]:
conv_by_type = campaign_df.groupby('Campaign_Type')['Conversion_Rate'].agg(['mean','median','count']).round(3)
conv_by_type.columns = ['Avg Conversion Rate', 'Median Conversion Rate', 'Campaign Count']
print(conv_by_type.sort_values('Avg Conversion Rate', ascending=False))

In [ ]:
plt.figure(figsize=(10, 5))
conv_by_type['Avg Conversion Rate'].sort_values(ascending=False).plot(kind='bar', color='steelblue')
plt.title('Average Conversion Rate by Campaign Type', fontsize=14)
plt.xlabel('Campaign Type')
plt.ylabel('Avg Conversion Rate')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

##Conversion Rate by Channel

In [ ]:
conv_by_channel = campaign_df.groupby('Channel_Used')['Conversion_Rate'].agg(['mean','count']).round(3)
conv_by_channel.columns = ['Avg Conversion Rate', 'Count']
print(conv_by_channel.sort_values('Avg Conversion Rate', ascending=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

conv_by_channel['Avg Conversion Rate'].sort_values(ascending=False).plot(
    kind='bar', ax=axes[0], color='darkorange')
axes[0].set_title('Avg Conversion Rate by Channel', fontsize=12)
axes[0].set_ylabel('Avg Conversion Rate')
axes[0].tick_params(axis='x', rotation=30)

campaign_df.groupby('Channel_Used')['Clicks'].sum().sort_values(ascending=False).plot(
    kind='bar', ax=axes[1], color='teal')
axes[1].set_title('Total Clicks by Channel', fontsize=12)
axes[1].set_ylabel('Total Clicks')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

##ROI Analysis

In [ ]:
roi_by_type = campaign_df.groupby('Campaign_Type')['ROI'].agg(['mean','median']).round(3)
roi_by_type.columns = ['Avg ROI', 'Median ROI']
print(roi_by_type.sort_values('Avg ROI', ascending=False))

In [ ]:
roi_by_channel = campaign_df.groupby('Channel_Used')['ROI'].mean().sort_values(ascending=False).round(3)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

roi_by_type['Avg ROI'].sort_values(ascending=False).plot(kind='bar', ax=axes[0], color='purple')
axes[0].set_title('Avg ROI by Campaign Type', fontsize=12)
axes[0].set_ylabel('Avg ROI')
axes[0].tick_params(axis='x', rotation=30)

roi_by_channel.plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Avg ROI by Channel', fontsize=12)
axes[1].set_ylabel('Avg ROI')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

##Acquisition Cost Analysis

In [ ]:
cost_by_type = campaign_df.groupby('Campaign_Type')['Acquisition_Cost'].agg(['mean','median']).round(2)
cost_by_type.columns = ['Avg Cost', 'Median Cost']
print(cost_by_type.sort_values('Avg Cost', ascending=False))

In [ ]:
# ROI vs Acquisition Cost scatter
plt.figure(figsize=(9, 6))
scatter = plt.scatter(
    campaign_df['Acquisition_Cost'],
    campaign_df['ROI'],
    c=campaign_df['Conversion_Rate'],
    cmap='viridis', alpha=0.6, s=30
)
plt.colorbar(scatter, label='Conversion Rate')
plt.title('ROI vs. Acquisition Cost (colored by Conversion Rate)', fontsize=13)
plt.xlabel('Acquisition Cost')
plt.ylabel('ROI')
plt.tight_layout()
plt.show()

##Customer Segment Analysis

In [ ]:
seg_metrics = campaign_df.groupby('Customer_Segment').agg(
    Avg_Conversion=('Conversion_Rate', 'mean'),
    Avg_ROI=('ROI', 'mean'),
    Avg_Engagement=('Engagement_Score', 'mean'),
    Count=('Campaign_ID', 'count')
).round(3)
print(seg_metrics.sort_values('Avg_Conversion', ascending=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

seg_metrics['Avg_Conversion'].sort_values(ascending=False).plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Avg Conversion Rate
by Customer Segment', fontsize=11)
axes[0].tick_params(axis='x', rotation=30)

seg_metrics['Avg_ROI'].sort_values(ascending=False).plot(kind='bar', ax=axes[1], color='darkorange')
axes[1].set_title('Avg ROI
by Customer Segment', fontsize=11)
axes[1].tick_params(axis='x', rotation=30)

seg_metrics['Avg_Engagement'].sort_values(ascending=False).plot(kind='bar', ax=axes[2], color='teal')
axes[2].set_title('Avg Engagement Score
by Customer Segment', fontsize=11)
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

##Correlation Heatmap

In [ ]:
numeric_cols = ['Clicks', 'Impressions', 'Engagement_Score', 'Conversion_Rate', 'Acquisition_Cost', 'ROI']
corr = campaign_df[numeric_cols].corr()

plt.figure(figsize=(9, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix — Campaign Metrics', fontsize=13)
plt.tight_layout()
plt.show()

#Summary

##Key Findings

- Campaign dataset is clean with no missing values across all fields
- Conversion rates and ROI differ meaningfully by **campaign type** and **channel** — not all channels are equal performers
- The correlation matrix reveals that **Engagement Score** is a stronger predictor of Conversion Rate than raw Click or Impression volume
- Acquisition cost does not directly predict ROI — some high-cost campaigns return lower ROI, pointing to targeting efficiency gaps
- Customer segment performance varies; certain segments show higher conversion but lower engagement, suggesting they respond better to direct-response messaging

##Recommendation

Prioritize channels with the highest conversion-to-cost ratio. Engagement score is a strong leading indicator — campaigns that drive engagement early are more likely to convert. Segment-specific creative strategies should be tested to close the gap between high-engagement and high-conversion segments.
